In [1]:
!nvidia-smi

Wed Aug 12 16:34:22 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   51C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!pip install -q transformers datasets torch torchvision

In [3]:
from datasets import load_dataset

dataset = load_dataset("nielsr/funsd")

README.md:   0%|          | 0.00/755 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 12.3MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 4.38MB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/149 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/50 [00:00<?, ? examples/s]

In [4]:
dataset

DatasetDict({
    train: Dataset({
        features: ['id', 'words', 'bboxes', 'ner_tags', 'image'],
        num_rows: 149
    })
    test: Dataset({
        features: ['id', 'words', 'bboxes', 'ner_tags', 'image'],
        num_rows: 50
    })
})

In [5]:
print(dataset["train"].features)

{'id': Value('string'), 'words': List(Value('string')), 'bboxes': List(List(Value('int64'))), 'ner_tags': List(ClassLabel(names=['O', 'B-HEADER', 'I-HEADER', 'B-QUESTION', 'I-QUESTION', 'B-ANSWER', 'I-ANSWER'])), 'image': Image(mode=None, decode=True)}


In [6]:
label_names = dataset["train"].features["ner_tags"].feature.names
print(label_names)

['O', 'B-HEADER', 'I-HEADER', 'B-QUESTION', 'I-QUESTION', 'B-ANSWER', 'I-ANSWER']


In [7]:
from transformers import AutoProcessor

processor = AutoProcessor.from_pretrained(
    "microsoft/layoutlm-base-uncased",
    apply_ocr=False
)

tokenizer_config.json:   0%|          | 0.00/170 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/606 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

In [8]:
def tokenize_and_align_labels(example):
    encoding = processor(
        images=example["image"],
        text=example["words"],
        boxes=example["bboxes"],
        is_split_into_words=True,
        truncation=True,
        padding="max_length",
        max_length=512
    )

    word_ids = encoding.word_ids()

    labels = []
    for word_id in word_ids:
        if word_id is None:
            labels.append(-100)
        else:
            labels.append(example["ner_tags"][word_id])

    encoding["labels"] = labels

    return encoding


tokenized_dataset = dataset.map(
    tokenize_and_align_labels,
    batched=False,
    remove_columns=dataset["train"].column_names
)

Map:   0%|          | 0/149 [00:00<?, ? examples/s]

Map:   0%|          | 0/50 [00:00<?, ? examples/s]

In [9]:
print(len(tokenized_dataset["train"][0]["input_ids"]))
print(len(tokenized_dataset["train"][0]["labels"]))

512
512


In [10]:
from transformers import AutoModelForTokenClassification

model = AutoModelForTokenClassification.from_pretrained(
    "microsoft/layoutlm-base-uncased",
    num_labels=len(label_names)
)

model.safetensors: reconstructing file:   0%|          |  0.00B /  451MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/203 [00:00<?, ?it/s]

[transformers] LayoutLMForTokenClassification LOAD REPORT from: microsoft/layoutlm-base-uncased
Key               | Status  | 
------------------+---------+-
classifier.bias   | MISSING | 
classifier.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [11]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./documind_model",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    learning_rate=5e-5,
    logging_steps=10,
    save_strategy="epoch",
    report_to="none"
)

In [12]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
)

In [13]:
for key in tokenized_dataset["train"].column_names:
    value = tokenized_dataset["train"][0][key]
    if isinstance(value, list):
        print(key, len(value))

input_ids 512
token_type_ids 512
attention_mask 512
labels 512


In [14]:
trainer.train()

Step,Training Loss
10,1.841390
20,1.792740
30,1.728136
40,1.757039
50,1.721085
60,1.723677
70,1.669392
80,1.676035
90,1.643299
100,1.610485


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=114, training_loss=1.7008790509742604, metrics={'train_runtime': 85.7519, 'train_samples_per_second': 5.213, 'train_steps_per_second': 1.329, 'total_flos': 117615921638400.0, 'train_loss': 1.7008790509742604, 'epoch': 3.0})

In [15]:
# step 10 upto this completed and next we are going to step 11.

In [16]:
train_valid = tokenized_dataset["train"].train_test_split(
    test_size=0.2,
    seed=42
)

train_dataset = train_valid["train"]
validation_dataset = train_valid["test"]

In [17]:
eval_results = trainer.evaluate(
    eval_dataset=validation_dataset
)

print(eval_results)

Training Loss,Validation Loss,Step
1.588067,1.541349,114


{'eval_loss': 1.5413486957550049}


In [18]:
!pip install -q seqeval

In [19]:
from seqeval.metrics import precision_score, recall_score, f1_score

predictions = trainer.predict(validation_dataset)

pred_labels = predictions.predictions.argmax(-1)
true_labels = predictions.label_ids

true_predictions = []
true_targets = []

for pred, true in zip(pred_labels, true_labels):
    p = []
    t = []

    for pred_id, true_id in zip(pred, true):
        if true_id != -100:
            p.append(label_names[pred_id])
            t.append(label_names[true_id])

    true_predictions.append(p)
    true_targets.append(t)

print("Precision:", precision_score(true_targets, true_predictions))
print("Recall:", recall_score(true_targets, true_predictions))
print("F1 Score:", f1_score(true_targets, true_predictions))

Precision: 0.3870808678500986
Recall: 0.36716557530402244
F1 Score: 0.3768602976476236


In [20]:
model.save_pretrained("./documind_model")
processor.save_pretrained("./documind_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./documind_model/tokenizer_config.json', './documind_model/tokenizer.json')

In [21]:
from google.colab import files

uploaded = files.upload()

Saving CV.pdf(1).pdf to CV.pdf(1) (1).pdf


In [22]:
!pip install -q pdf2image
!apt-get -qq install poppler-utils

Selecting previously unselected package poppler-utils.
(Reading database ... 122492 files and directories currently installed.)
Preparing to unpack .../poppler-utils_22.02.0-2ubuntu0.13_amd64.deb ...
Unpacking poppler-utils (22.02.0-2ubuntu0.13) ...
Setting up poppler-utils (22.02.0-2ubuntu0.13) ...
Processing triggers for man-db (2.10.2-1) ...


In [23]:
from pdf2image import convert_from_path

pages = convert_from_path("CV.pdf(1) (1).pdf")

print("Pages:", len(pages))

Pages: 1


In [ ]:
# step 16 below

In [24]:
!apt-get -qq install tesseract-ocr
!pip install -q pytesseract

In [25]:
import pytesseract
from pytesseract import Output

image = pages[0]

ocr_data = pytesseract.image_to_data(
    image,
    output_type=Output.DICT
)

words = []
boxes = []

for i, text in enumerate(ocr_data["text"]):
    text = text.strip()

    if text:
        x = ocr_data["left"][i]
        y = ocr_data["top"][i]
        w = ocr_data["width"][i]
        h = ocr_data["height"][i]

        words.append(text)
        boxes.append([x, y, x + w, y + h])

print("Words detected:", len(words))
print(words[:20])

Words detected: 478
['KEERHTI', 'MM', 'Final', 'Year', 'Information', 'Technology', 'Student', 'mmkeerthi28@gmail.com', '-', '+91', '6381528240', 'https://www.linkedin.com/in/keerthi-mm-298880288', '—', 'https://leetcode.com/u/Keerthimuthu/', 'SUMMARY', 'IT', 'student', 'passionate', 'about', 'problem']


In [26]:
width, height = image.size

normalized_boxes = []

for box in boxes:
    x1, y1, x2, y2 = box

    normalized_boxes.append([
        int(x1 / width * 1000),
        int(y1 / height * 1000),
        int(x2 / width * 1000),
        int(y2 / height * 1000)
    ])

print(normalized_boxes[:5])

[[386, 42, 520, 60], [535, 42, 609, 60], [344, 71, 378, 80], [384, 72, 416, 80], [422, 71, 511, 80]]


In [30]:
import torch
inputs = processor(
    images=image,
    text=words,
    boxes=normalized_boxes,
    is_split_into_words=True,
    return_tensors="pt",
    truncation=True,
    padding="max_length",
    max_length=512
)

inputs = {k: v.to(model.device) for k, v in inputs.items()}

with torch.no_grad():
    outputs = model(**inputs)

predictions = outputs.logits.argmax(-1)

print("Prediction completed!")

Prediction completed!


In [31]:
predicted_labels = [
    label_names[pred]
    for pred in predictions[0][:len(words)]
]

for word, label in zip(words, predicted_labels):
    print(f"{word:20} → {label}")

KEERHTI              → I-QUESTION
MM                   → B-QUESTION
Final                → B-QUESTION
Year                 → B-QUESTION
Information          → B-QUESTION
Technology           → B-QUESTION
Student              → B-QUESTION
mmkeerthi28@gmail.com → B-QUESTION
-                    → B-QUESTION
+91                  → B-QUESTION
6381528240           → B-QUESTION
https://www.linkedin.com/in/keerthi-mm-298880288 → B-QUESTION
—                    → B-QUESTION
https://leetcode.com/u/Keerthimuthu/ → B-QUESTION
SUMMARY              → B-ANSWER
IT                   → B-ANSWER
student              → B-ANSWER
passionate           → B-QUESTION
about                → B-QUESTION
problem              → B-ANSWER
solving,             → B-ANSWER
driven               → B-ANSWER
to                   → B-ANSWER
build                → B-ANSWER
scalable             → B-ANSWER
systems              → B-ANSWER
that                 → B-ANSWER
translate            → B-ANSWER
ideas                → B-AN

In [32]:
from collections import defaultdict

extracted_data = defaultdict(list)

for word, label in zip(words, predicted_labels):
    if label != "O":
        extracted_data[label].append(word)

for label, items in extracted_data.items():
    print(f"\n{label}:")
    print(" ".join(items))


I-QUESTION:
KEERHTI impact.Adept and Machine an Precision, Question

B-QUESTION:
MM Final Year Information Technology Student mmkeerthi28@gmail.com - +91 6381528240 https://www.linkedin.com/in/keerthi-mm-298880288 — https://leetcode.com/u/Keerthimuthu/ passionate about logic, optimizing applications.Proven problem-

B-ANSWER:
SUMMARY IT student problem solving, driven to build scalable systems that translate ideas into realworld at crafting efficient system performance, and developing maintainable server-side solver through active participation in national-level hackathons like Sri

I-ANSWER:
clean, SIH,KPIT Sparkle, Techgium, turning challenges into innovative solutions. EDUCATION Krishna College Of Engineering and Technology , Coimbatore 2023 — 2027 Bachelor of Information Technology CGPA: 8.3/10 Professional Elective : Machine Learning TECHNICAL SKILLS Programming Languages: Python,Java,c++ Deep Learning & Al: PyTorch, TensorFlow, Hugging Face Transformers, CNN, RNN/LSTM, Computer 

In [33]:
structured_data = {
    "headers": [],
    "questions": [],
    "answers": []
}

for word, label in zip(words, predicted_labels):
    if "HEADER" in label:
        structured_data["headers"].append(word)
    elif "QUESTION" in label:
        structured_data["questions"].append(word)
    elif "ANSWER" in label:
        structured_data["answers"].append(word)

print(structured_data)

{'headers': [], 'questions': ['KEERHTI', 'MM', 'Final', 'Year', 'Information', 'Technology', 'Student', 'mmkeerthi28@gmail.com', '-', '+91', '6381528240', 'https://www.linkedin.com/in/keerthi-mm-298880288', '—', 'https://leetcode.com/u/Keerthimuthu/', 'passionate', 'about', 'impact.Adept', 'logic,', 'optimizing', 'applications.Proven', 'problem-', 'and', 'Machine', 'an', 'Precision,', 'Question'], 'answers': ['SUMMARY', 'IT', 'student', 'problem', 'solving,', 'driven', 'to', 'build', 'scalable', 'systems', 'that', 'translate', 'ideas', 'into', 'realworld', 'at', 'crafting', 'efficient', 'system', 'performance,', 'and', 'developing', 'clean,', 'maintainable', 'server-side', 'solver', 'through', 'active', 'participation', 'in', 'national-level', 'hackathons', 'like', 'SIH,KPIT', 'Sparkle,', 'Techgium,', 'turning', 'challenges', 'into', 'innovative', 'solutions.', 'EDUCATION', 'Sri', 'Krishna', 'College', 'Of', 'Engineering', 'and', 'Technology', ',', 'Coimbatore', '2023', '—', '2027', 'B

In [ ]:
# step 22

In [34]:
!pip install -q gradio

In [36]:
import gradio as gr
import torch
import pytesseract
from pytesseract import Output
from pdf2image import convert_from_path

def analyze_document(file):
    # 1. Get uploaded file path
    file_path = file.name if hasattr(file, "name") else file

    # 2. Convert PDF to image
    pages = convert_from_path(file_path)

    # Use first page for now
    image = pages[0]

    # 3. OCR
    ocr_data = pytesseract.image_to_data(
        image,
        output_type=Output.DICT
    )

    words = []
    boxes = []

    for i, text in enumerate(ocr_data["text"]):
        text = text.strip()

        if text:
            x = ocr_data["left"][i]
            y = ocr_data["top"][i]
            w = ocr_data["width"][i]
            h = ocr_data["height"][i]

            words.append(text)
            boxes.append([x, y, x + w, y + h])

    # 4. Normalize bounding boxes to 0-1000
    width, height = image.size

    normalized_boxes = []

    for x1, y1, x2, y2 in boxes:
        normalized_boxes.append([
            int(x1 / width * 1000),
            int(y1 / height * 1000),
            int(x2 / width * 1000),
            int(y2 / height * 1000)
        ])

    # 5. Prepare input for LayoutLM
    inputs = processor(
        images=image,
        text=words,
        boxes=normalized_boxes,
        is_split_into_words=True,
        return_tensors="pt",
        truncation=True,
        padding="max_length",
        max_length=512
    )

    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    # 6. Model prediction
    with torch.no_grad():
        outputs = model(**inputs)

    predictions = outputs.logits.argmax(-1)[0].cpu().tolist()

    # 7. Convert predictions to labels
    word_ids = inputs["input_ids"].new_tensor(
        processor(
            images=image,
            text=words,
            boxes=normalized_boxes,
            is_split_into_words=True,
            truncation=True,
            padding="max_length",
            max_length=512
        )["input_ids"]
    )

    # Get word mapping
    encoding = processor(
        images=image,
        text=words,
        boxes=normalized_boxes,
        is_split_into_words=True,
        truncation=True,
        padding="max_length",
        max_length=512
    )

    word_ids = encoding.word_ids()

    results = []

    for token_index, word_id in enumerate(word_ids):
        if word_id is not None and word_id < len(words):
            label = label_names[predictions[token_index]]
            results.append(f"{words[word_id]} → {label}")

    return "\n".join(results)


demo = gr.Interface(
    fn=analyze_document,
    inputs=gr.File(
        label="Upload Document",
        file_types=[".pdf", ".png", ".jpg", ".jpeg"]
    ),
    outputs=gr.Textbox(
        label="Extracted Information",
        lines=20
    ),
    title="DocuMind AI",
    description="AI-powered document understanding and information extraction"
)

demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://012057b9a53e32a14a.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [37]:
def analyze_document(file):
    file_path = file.name if hasattr(file, "name") else file

    # Convert all PDF pages to images
    pages = convert_from_path(file_path)

    final_output = {
        "HEADERS": [],
        "QUESTIONS": [],
        "ANSWERS": []
    }

    for page_number, image in enumerate(pages, start=1):

        # OCR
        ocr_data = pytesseract.image_to_data(
            image,
            output_type=Output.DICT
        )

        words = []
        boxes = []

        for i, text in enumerate(ocr_data["text"]):
            text = text.strip()

            if text:
                x = ocr_data["left"][i]
                y = ocr_data["top"][i]
                w = ocr_data["width"][i]
                h = ocr_data["height"][i]

                words.append(text)
                boxes.append([x, y, x + w, y + h])

        if not words:
            continue

        # Normalize boxes
        width, height = image.size

        normalized_boxes = [
            [
                int(x1 / width * 1000),
                int(y1 / height * 1000),
                int(x2 / width * 1000),
                int(y2 / height * 1000)
            ]
            for x1, y1, x2, y2 in boxes
        ]

        # LayoutLM input
        encoding = processor(
            images=image,
            text=words,
            boxes=normalized_boxes,
            is_split_into_words=True,
            truncation=True,
            padding="max_length",
            max_length=512,
            return_tensors="pt"
        )

        word_ids = encoding.word_ids()

        inputs = {
            k: v.to(model.device)
            for k, v in encoding.items()
            if hasattr(v, "to")
        }

        # Prediction
        with torch.no_grad():
            outputs = model(**inputs)

        predictions = outputs.logits.argmax(-1)[0].cpu().tolist()

        # Collect results
        for token_index, word_id in enumerate(word_ids):

            if word_id is None or word_id >= len(words):
                continue

            label = label_names[predictions[token_index]]
            word = words[word_id]

            if label == "B-HEADER" or label == "I-HEADER":
                final_output["HEADERS"].append(word)

            elif label == "B-QUESTION" or label == "I-QUESTION":
                final_output["QUESTIONS"].append(word)

            elif label == "B-ANSWER" or label == "I-ANSWER":
                final_output["ANSWERS"].append(word)

    # Display clean result
    result = "=== DOCUMIND AI RESULTS ===\n\n"

    result += "HEADERS:\n"
    result += " ".join(final_output["HEADERS"]) + "\n\n"

    result += "QUESTIONS:\n"
    result += " ".join(final_output["QUESTIONS"]) + "\n\n"

    result += "ANSWERS:\n"
    result += " ".join(final_output["ANSWERS"])

    return result


demo = gr.Interface(
    fn=analyze_document,
    inputs=gr.File(
        label="Upload Document",
        file_types=[".pdf", ".png", ".jpg", ".jpeg"]
    ),
    outputs=gr.Textbox(
        label="Extracted Information",
        lines=25
    ),
    title="DocuMind AI",
    description="AI-powered document understanding and information extraction"
)

demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://80707e72bb6b832f1f.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
# step 25 specifically for my cv

In [38]:
!pip install -q spacy

In [39]:
import re

def extract_cv_fields(text):
    data = {
        "Name": "",
        "Email": "",
        "Phone": "",
        "Skills": []
    }

    # Email
    email = re.search(r'[\w\.-]+@[\w\.-]+\.\w+', text)
    if email:
        data["Email"] = email.group()

    # Phone
    phone = re.search(r'\b(?:\+91[-\s]?)?[6-9]\d{9}\b', text)
    if phone:
        data["Phone"] = phone.group()

    # Skills
    skills_list = [
        "Python", "Java", "C++", "JavaScript",
        "React", "Node.js", "Machine Learning",
        "Deep Learning", "SQL", "MongoDB",
        "TensorFlow", "PyTorch"
    ]

    for skill in skills_list:
        if skill.lower() in text.lower():
            data["Skills"].append(skill)

    return data

In [40]:
def analyze_cv(file):
    raw_result = analyze_document(file)

    extracted = extract_cv_fields(raw_result)

    return (
        f"Name: {extracted['Name']}\n\n"
        f"Email: {extracted['Email']}\n\n"
        f"Phone: {extracted['Phone']}\n\n"
        f"Skills: {', '.join(extracted['Skills'])}"
    )


demo = gr.Interface(
    fn=analyze_cv,
    inputs=gr.File(
        label="Upload CV",
        file_types=[".pdf", ".png", ".jpg", ".jpeg"]
    ),
    outputs=gr.Textbox(
        label="Extracted CV Information",
        lines=15
    ),
    title="DocuMind AI",
    description="Upload a CV to extract important information"
)

demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://edfa11fefb1090645d.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
# general purposes

In [41]:
def analyze_general_document(file):
    raw_result = analyze_document(file)

    return raw_result


demo = gr.Interface(
    fn=analyze_general_document,
    inputs=gr.File(
        label="Upload Any Document",
        file_types=[".pdf", ".png", ".jpg", ".jpeg"]
    ),
    outputs=gr.Textbox(
        label="Document Information",
        lines=25
    ),
    title="DocuMind AI",
    description="Upload a document to extract and understand its information"
)

demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://488406dd00cd3fd8a7.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [42]:
clean_words = []
clean_boxes = []
seen = set()

for word, box in zip(words, boxes):
    key = word.lower()

    if key not in seen:
        seen.add(key)
        clean_words.append(word)
        clean_boxes.append(box)

words = clean_words
boxes = clean_boxes

print("Words after cleaning:", len(words))
print(words[:30])

Words after cleaning: 309
['KEERHTI', 'MM', 'Final', 'Year', 'Information', 'Technology', 'Student', 'mmkeerthi28@gmail.com', '-', '+91', '6381528240', 'https://www.linkedin.com/in/keerthi-mm-298880288', '—', 'https://leetcode.com/u/Keerthimuthu/', 'SUMMARY', 'IT', 'passionate', 'about', 'problem', 'solving,', 'driven', 'to', 'build', 'scalable', 'systems', 'that', 'translate', 'ideas', 'into', 'realworld']


In [43]:
words = []
boxes = []

for i, text in enumerate(ocr_data["text"]):
    text = text.strip()
    confidence = float(ocr_data["conf"][i])

    if text and confidence >= 50:
        x = ocr_data["left"][i]
        y = ocr_data["top"][i]
        w = ocr_data["width"][i]
        h = ocr_data["height"][i]

        words.append(text)
        boxes.append([x, y, x + w, y + h])

print("Reliable words:", len(words))
print(words[:30])

Reliable words: 467
['KEERHTI', 'MM', 'Final', 'Year', 'Information', 'Technology', 'Student', 'mmkeerthi28@gmail.com', '-', 'https://www.linkedin.com/in/keerthi-mm-298880288', '—', 'https://leetcode.com/u/Keerthimuthu/', 'SUMMARY', 'IT', 'student', 'passionate', 'about', 'problem', 'solving,', 'driven', 'to', 'build', 'scalable', 'systems', 'that', 'translate', 'ideas', 'into', 'realworld', 'impact.Adept']


In [50]:
import torch

# Create LayoutLM encoding
encoding = processor(
    images=image,
    text=words,
    boxes=normalized_boxes,
    is_split_into_words=True,
    truncation=True,
    padding="max_length",
    max_length=512,
    return_tensors="pt"
)

# Keep word mapping before converting to dictionary
word_ids = encoding.word_ids()

# Send inputs to GPU/CPU
inputs = {
    k: v.to(model.device)
    for k, v in encoding.items()
    if hasattr(v, "to")
}

# Model prediction
with torch.no_grad():
    outputs = model(**inputs)

predictions = outputs.logits.argmax(-1)[0].cpu().tolist()

# One prediction per word
word_predictions = {}

for token_index, word_id in enumerate(word_ids):
    if word_id is not None and word_id < len(words):
        if word_id not in word_predictions:
            word_predictions[word_id] = label_names[predictions[token_index]]

# Clean output
clean_results = []

for word_id, label in word_predictions.items():
    clean_results.append(
        f"{words[word_id]} → {label}"
    )

print("\n".join(clean_results))

KEERHTI → B-QUESTION
MM → B-QUESTION
Final → B-QUESTION
Year → B-QUESTION
Information → B-QUESTION
Technology → B-QUESTION
Student → B-QUESTION
mmkeerthi28@gmail.com → B-QUESTION
- → B-QUESTION
https://www.linkedin.com/in/keerthi-mm-298880288 → B-QUESTION
— → B-QUESTION
https://leetcode.com/u/Keerthimuthu/ → B-QUESTION
SUMMARY → I-ANSWER
IT → I-ANSWER
student → I-ANSWER
passionate → I-ANSWER
about → I-ANSWER
problem → I-ANSWER
solving, → I-ANSWER
driven → I-ANSWER
to → I-ANSWER
build → I-ANSWER
scalable → I-ANSWER
systems → I-ANSWER
that → I-ANSWER
translate → I-ANSWER
ideas → I-ANSWER
into → I-ANSWER
realworld → I-ANSWER
impact.Adept → I-ANSWER
at → I-ANSWER
crafting → I-ANSWER
efficient → I-ANSWER
system → I-ANSWER
logic, → I-ANSWER
optimizing → I-ANSWER
performance, → I-QUESTION
and → I-ANSWER
developing → I-ANSWER
clean, → I-ANSWER
maintainable → I-ANSWER
server-side → I-ANSWER
applications.Proven → I-ANSWER
problem- → I-ANSWER
solver → I-ANSWER
through → I-ANSWER
active → I-ANSWER

In [51]:
from collections import defaultdict

grouped_results = defaultdict(list)

for word_id, label in word_predictions.items():
    grouped_results[label].append(words[word_id])

for label, items in grouped_results.items():
    print(f"\n{label}:")
    print(" ".join(items))


B-QUESTION:
KEERHTI MM Final Year Information Technology Student mmkeerthi28@gmail.com - https://www.linkedin.com/in/keerthi-mm-298880288 — https://leetcode.com/u/Keerthimuthu/

I-ANSWER:
SUMMARY IT student passionate about problem solving, driven to build scalable systems that translate ideas into realworld impact.Adept at crafting efficient system logic, optimizing and developing clean, maintainable server-side applications.Proven problem- solver through active participation in national-level hackathons like SIH,KPIT Sparkle, and Techgium, turning challenges into innovative solutions. EDUCATION Sri Krishna College Of Engineering and Technology , Coimbatore 2023 — 2027 Bachelor of Information Technology CGPA: 8.3/10 Professional Elective : Machine Learning TECHNICAL SKILLS Programming Languages: Python,Java,c++ Deep Learning & Al: PyTorch, TensorFlow, Hugging Face Transformers, CNN, RNN/LSTM, Computer Vision, LLMs Machine Learning: Scikit-learn, NumPy, Pandas, OpenCV, Matplotlib Fram

In [52]:
import json

structured_output = {
    "headers": grouped_results.get("B-HEADER", []) + grouped_results.get("I-HEADER", []),
    "questions": grouped_results.get("B-QUESTION", []) + grouped_results.get("I-QUESTION", []),
    "answers": grouped_results.get("B-ANSWER", []) + grouped_results.get("I-ANSWER", [])
}

print(json.dumps(structured_output, indent=4))

{
    "headers": [],
    "questions": [
        "KEERHTI",
        "MM",
        "Final",
        "Year",
        "Information",
        "Technology",
        "Student",
        "mmkeerthi28@gmail.com",
        "-",
        "https://www.linkedin.com/in/keerthi-mm-298880288",
        "\u2014",
        "https://leetcode.com/u/Keerthimuthu/",
        "performance,"
    ],
    "answers": [
        "SUMMARY",
        "IT",
        "student",
        "passionate",
        "about",
        "problem",
        "solving,",
        "driven",
        "to",
        "build",
        "scalable",
        "systems",
        "that",
        "translate",
        "ideas",
        "into",
        "realworld",
        "impact.Adept",
        "at",
        "crafting",
        "efficient",
        "system",
        "logic,",
        "optimizing",
        "and",
        "developing",
        "clean,",
        "maintainable",
        "server-side",
        "applications.Proven",
        "problem-",
        "sol

In [54]:
def documind_app(file):
    file_path = file.name if hasattr(file, "name") else file

    pages = convert_from_path(file_path)

    final_results = []

    for page_no, image in enumerate(pages, start=1):

        # OCR
        ocr_data = pytesseract.image_to_data(
            image,
            output_type=Output.DICT
        )

        words = []
        boxes = []

        for i, text in enumerate(ocr_data["text"]):
            text = text.strip()
            confidence = float(ocr_data["conf"][i])

            if text and confidence >= 50:
                x = ocr_data["left"][i]
                y = ocr_data["top"][i]
                w = ocr_data["width"][i]
                h = ocr_data["height"][i]

                words.append(text)
                boxes.append([x, y, x + w, y + h])

        if not words:
            continue

        # Normalize boxes
        width, height = image.size

        normalized_boxes = [
            [
                int(x1 / width * 1000),
                int(y1 / height * 1000),
                int(x2 / width * 1000),
                int(y2 / height * 1000)
            ]
            for x1, y1, x2, y2 in boxes
        ]

        # LayoutLM encoding
        encoding = processor(
            images=image,
            text=words,
            boxes=normalized_boxes,
            is_split_into_words=True,
            truncation=True,
            padding="max_length",
            max_length=512,
            return_tensors="pt"
        )

        word_ids = encoding.word_ids()

        inputs = {
            k: v.to(model.device)
            for k, v in encoding.items()
            if hasattr(v, "to")
        }

        # Prediction
        with torch.no_grad():
            outputs = model(**inputs)

        predictions = outputs.logits.argmax(-1)[0].cpu().tolist()

        # One prediction per word
        seen_words = set()

        page_results = []

        for token_index, word_id in enumerate(word_ids):

            if word_id is None or word_id >= len(words):
                continue

            if word_id in seen_words:
                continue

            seen_words.add(word_id)

            word = words[word_id]
            label = label_names[predictions[token_index]]

            if label != "O":
                page_results.append(
                    f"{word} → {label}"
                )

        if page_results:
            final_results.append(
                f"--- Page {page_no} ---\n" +
                "\n".join(page_results)
            )

    if not final_results:
        return "No structured information detected."

    return "\n\n".join(final_results)


demo = gr.Interface(
    fn=documind_app,
    inputs=gr.File(
        label="Upload Document",
        file_types=[".pdf", ".png", ".jpg", ".jpeg"]
    ),
    outputs=gr.Textbox(
        label="DocuMind AI Results",
        lines=30
    ),
    title="DocuMind AI",
    description="Upload a document for AI-powered information extraction"
)

demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://0a8e8a41776b8b2f17.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [63]:
import torch
import gradio as gr
import pytesseract

from collections import defaultdict
from pytesseract import Output
from pdf2image import convert_from_path


def documind_app(file):

    file_path = file.name if hasattr(file, "name") else file

    pages = convert_from_path(file_path)

    final_output = []

    for page_no, image in enumerate(pages, start=1):

        # -----------------------------
        # 1. OCR
        # -----------------------------
        ocr = pytesseract.image_to_data(
            image,
            output_type=Output.DICT
        )

        words = []
        boxes = []

        for i, text in enumerate(ocr["text"]):

            text = text.strip()

            try:
                confidence = float(ocr["conf"][i])
            except:
                confidence = 0

            if not text or confidence < 50:
                continue

            x = ocr["left"][i]
            y = ocr["top"][i]
            w = ocr["width"][i]
            h = ocr["height"][i]

            box = [x, y, x + w, y + h]

            words.append(text)
            boxes.append(box)

        if not words:
            continue

        # -----------------------------
        # 2. Remove exact duplicate
        #    OCR detections
        # -----------------------------
        unique_words = []
        unique_boxes = []
        seen = set()

        for word, box in zip(words, boxes):

            key = (
                word.lower(),
                box[0],
                box[1],
                box[2],
                box[3]
            )

            if key not in seen:
                seen.add(key)
                unique_words.append(word)
                unique_boxes.append(box)

        words = unique_words
        boxes = unique_boxes

        # -----------------------------
        # 3. Normalize bounding boxes
        # -----------------------------
        width, height = image.size

        normalized_boxes = []

        for x1, y1, x2, y2 in boxes:

            normalized_boxes.append([
                max(0, min(1000, int(x1 / width * 1000))),
                max(0, min(1000, int(y1 / height * 1000))),
                max(0, min(1000, int(x2 / width * 1000))),
                max(0, min(1000, int(y2 / height * 1000)))
            ])

        # -----------------------------
        # 4. LayoutLM processor
        # -----------------------------
        encoding = processor(
            images=image,
            text=words,
            boxes=normalized_boxes,
            is_split_into_words=True,
            truncation=True,
            padding="max_length",
            max_length=512,
            return_tensors="pt"
        )

        word_ids = encoding.word_ids()

        # -----------------------------
        # 5. Move input to GPU/CPU
        # -----------------------------
        inputs = {
            k: v.to(model.device)
            for k, v in encoding.items()
            if hasattr(v, "to")
        }

        # -----------------------------
        # 6. Model prediction
        # -----------------------------
        with torch.no_grad():

            outputs = model(**inputs)

        predictions = outputs.logits.argmax(
            dim=-1
        )[0].cpu().tolist()

        # -----------------------------
        # 7. One prediction per word
        # -----------------------------
        word_predictions = {}

        for token_index, word_id in enumerate(word_ids):

            if word_id is None:
                continue

            if word_id >= len(words):
                continue

            if word_id not in word_predictions:

                word_predictions[word_id] = (
                    label_names[predictions[token_index]]
                )

        # -----------------------------
        # 8. Group results
        # -----------------------------
        grouped = defaultdict(list)

        for word_id, label in word_predictions.items():

            word = words[word_id]

            if label != "O":
                grouped[label].append(word)

        # -----------------------------
        # 9. Create page output
        # -----------------------------
        page_output = f"===== PAGE {page_no} =====\n\n"

        for label in [
            "B-HEADER",
            "I-HEADER",
            "B-QUESTION",
            "I-QUESTION",
            "B-ANSWER",
            "I-ANSWER"
        ]:

            if grouped[label]:

                page_output += f"{label}:\n"

                page_output += " ".join(
                    grouped[label]
                )

                page_output += "\n\n"

        final_output.append(page_output)

    if not final_output:
        return "No information detected."

    return "\n".join(final_output)


# -----------------------------
# Gradio Interface
# -----------------------------

demo = gr.Interface(
    fn=documind_app,

    inputs=gr.File(
        label="Upload Document",
        file_types=[
            ".pdf",
            ".png",
            ".jpg",
            ".jpeg"
        ]
    ),

    outputs=gr.Textbox(
        label="DocuMind AI Result",
        lines=30
    ),

    title="DocuMind AI",

    description=(
        "AI-powered document understanding "
        "and information extraction"
    )
)

demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://84a3247b9e4bc6ccf4.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [68]:
import torch
import gradio as gr
import pytesseract
import json

from collections import defaultdict
from pytesseract import Output
from pdf2image import convert_from_path


def documind_app(file):

    file_path = file.name if hasattr(file, "name") else file

    pages = convert_from_path(file_path)

    final_output = []

    for page_no, image in enumerate(pages, start=1):

        # -----------------------------
        # 1. OCR
        # -----------------------------
        ocr = pytesseract.image_to_data(
            image,
            output_type=Output.DICT
        )

        words = []
        boxes = []

        for i, text in enumerate(ocr["text"]):

            text = text.strip()

            try:
                confidence = float(ocr["conf"][i])
            except:
                confidence = 0

            if not text or confidence < 50:
                continue

            x = ocr["left"][i]
            y = ocr["top"][i]
            w = ocr["width"][i]
            h = ocr["height"][i]

            box = [x, y, x + w, y + h]

            words.append(text)
            boxes.append(box)

        if not words:
            continue

        # -----------------------------
        # 2. Remove exact duplicates
        # -----------------------------
        unique_words = []
        unique_boxes = []
        seen = set()

        for word, box in zip(words, boxes):

            key = (
                word.lower(),
                box[0],
                box[1],
                box[2],
                box[3]
            )

            if key not in seen:
                seen.add(key)
                unique_words.append(word)
                unique_boxes.append(box)

        words = unique_words
        boxes = unique_boxes

        # -----------------------------
        # 3. Normalize bounding boxes
        # -----------------------------
        width, height = image.size

        normalized_boxes = []

        for x1, y1, x2, y2 in boxes:

            normalized_boxes.append([
                max(0, min(1000, int(x1 / width * 1000))),
                max(0, min(1000, int(y1 / height * 1000))),
                max(0, min(1000, int(x2 / width * 1000))),
                max(0, min(1000, int(y2 / height * 1000)))
            ])

        # -----------------------------
        # 4. LayoutLM processor
        # -----------------------------
        encoding = processor(
            images=image,
            text=words,
            boxes=normalized_boxes,
            is_split_into_words=True,
            truncation=True,
            padding="max_length",
            max_length=512,
            return_tensors="pt"
        )

        word_ids = encoding.word_ids()

        # -----------------------------
        # 5. Move input to GPU/CPU
        # -----------------------------
        inputs = {
            k: v.to(model.device)
            for k, v in encoding.items()
            if hasattr(v, "to")
        }

        # -----------------------------
        # 6. Model prediction
        # -----------------------------
        with torch.no_grad():

            outputs = model(**inputs)

        predictions = outputs.logits.argmax(
            dim=-1
        )[0].cpu().tolist()

        # -----------------------------
        # 7. One prediction per word
        # -----------------------------
        word_predictions = {}

        for token_index, word_id in enumerate(word_ids):

            if word_id is None:
                continue

            if word_id >= len(words):
                continue

            if word_id not in word_predictions:

                word_predictions[word_id] = (
                    label_names[predictions[token_index]]
                )

        # -----------------------------
        # 8. Group results
        # -----------------------------
        grouped = defaultdict(list)

        for word_id, label in word_predictions.items():

            word = words[word_id]

            if label != "O":
                grouped[label].append(word)

        # -----------------------------
        # 9. Structured JSON
        # -----------------------------
        page_data = {
            "page": page_no,
            "fields": {}
        }

        for label in [
            "B-HEADER",
            "I-HEADER",
            "B-QUESTION",
            "I-QUESTION",
            "B-ANSWER",
            "I-ANSWER"
        ]:

            if grouped[label]:

                page_data["fields"][label] = " ".join(
                    grouped[label]
                )

        final_output.append(page_data)

    # -----------------------------
    # 10. Return JSON
    # -----------------------------
    if not final_output:
        return json.dumps(
            {
                "message": "No information detected."
            },
            indent=4
        )

    return json.dumps(
        final_output,
        indent=4
    )


# -----------------------------
# Gradio Interface
# -----------------------------

demo = gr.Interface(
    fn=documind_app,

    inputs=gr.File(
        label="Upload Document",
        file_types=[
            ".pdf",
            ".png",
            ".jpg",
            ".jpeg"
        ]
    ),

    outputs=gr.Code(
        label="DocuMind AI - JSON Output",
        language="json"
    ),

    title="DocuMind AI",

    description=(
        "AI-powered document understanding "
        "and information extraction"
    )
)

demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://22db3c0cbb574f80a4.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [69]:
!git clone https://github.com/mmkeerthi28-blip/DocuMind-AI.git

Cloning into 'DocuMind-AI'...
remote: Enumerating objects: 3, done.
remote: Counting objects: 100% (3/3), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 3 (delta 0), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (3/3), done.


In [71]:
!git config --global user.name "mmkeerthi28-blip"
!git config --global user.email "mmkeerthi28@gmail.com"

In [72]:
%cd /content/DocuMind-AI

!git add .
!git commit -m "Add DocuMind AI project"

/content/DocuMind-AI
On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean


In [73]:
%cd /content/DocuMind-AI
!ls

/content/DocuMind-AI
README.md


In [74]:
!git status

On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean


In [75]:
!ls /content/*.ipynb

ls: cannot access '/content/*.ipynb': No such file or directory
